In [ ]:
# v7_model — Major feature expansion + target-specific optimization
#
# Key improvements over v6:
#   1. NEW: Red + Blue Landsat bands → NDVI, EVI, turbidity proxy, BSI, AWEI, NDBI
#   2. NEW: Spatial proxy features (cluster-level stats, lat/lon interactions, coast distance)
#   3. NEW: DRP log-transform (Dissolved Reactive Phosphorus is heavily right-skewed)
#   4. NEW: Target-specific feature importance filtering
#   5. KEPT: Multi-seed LightGBM ensemble, sequential target chain, GroupKFold spatial CV

import warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, gc, os
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score
from sklearn.cluster import KMeans
from lightgbm import LGBMRegressor
import lightgbm as lgb

# ====================== CONFIG ======================
WORKDIR = Path(".")
SEED = 42
SPATIAL_ROUND = 3
N_SPLITS = 5
N_CLUSTERS = 20
N_SEEDS = 3
SEEDS = [SEED + i * 111 for i in range(N_SEEDS)]  # [42, 153, 264]

TRAIN_LABELS_CSV = WORKDIR / "water_quality_training_dataset.csv"
LANDSAT_TRAIN_CSV = WORKDIR / "landsat_features_training.csv"
LANDSAT_VAL_CSV = WORKDIR / "landsat_features_validation.csv"
TERRACLIMATE_TRAIN_CSV = WORKDIR / "terraclimate_features_training.csv"
TERRACLIMATE_VAL_CSV = WORKDIR / "terraclimate_features_validation.csv"
SUBMISSION_TEMPLATE_CSV = WORKDIR / "submission_template.csv"

OUT_LOCAL = WORKDIR / "submission.csv"
OUT_TMP = "/tmp/submission.csv"

TARGETS = ["Total Alkalinity", "Electrical Conductance", "Dissolved Reactive Phosphorus"]

# Noise features to drop
DROP_FEATURES = ["swe"]  # All zeros in South Africa

# ====================== SANITY CHECKS ======================
for p in [TRAIN_LABELS_CSV, LANDSAT_TRAIN_CSV, LANDSAT_VAL_CSV,
          TERRACLIMATE_TRAIN_CSV, TERRACLIMATE_VAL_CSV, SUBMISSION_TEMPLATE_CSV]:
    print(p.name, "exists:", p.exists())
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {p}")

print("Working directory:", os.getcwd())

# ====================== HELPERS ======================
def parse_date(s):
    return pd.to_datetime(s, format="mixed", dayfirst=True)

def add_month_key_and_round(df, date_col="Sample Date", round_digits=SPATIAL_ROUND):
    df = df.copy()
    df[date_col] = parse_date(df[date_col])
    df["month_ts"] = df[date_col].dt.to_period("M").dt.to_timestamp()
    df["lat_round"] = df["Latitude"].round(round_digits)
    df["lon_round"] = df["Longitude"].round(round_digits)
    return df

# ====================== FEATURE ENGINEERING ======================
def engineer_landsat_features(df):
    """Engineer features from Landsat bands including RED and BLUE."""
    out = df.copy()
    eps = 1e-10

    # Core 4 bands (always available)
    required_4 = ["nir", "green", "swir16", "swir22"]
    if not all(c in out.columns for c in required_4):
        print("Warning: Missing core Landsat columns, skipping engineering")
        return out

    nir = out["nir"].astype(float)
    green = out["green"].astype(float)
    swir16 = out["swir16"].astype(float)
    swir22 = out["swir22"].astype(float)

    # Check for red and blue bands (NEW in v7)
    has_red = "red" in out.columns
    has_blue = "blue" in out.columns
    if has_red:
        red = out["red"].astype(float)
        print(f"  RED band available: {red.notna().sum()}/{len(red)} non-null")
    if has_blue:
        blue = out["blue"].astype(float)
        print(f"  BLUE band available: {blue.notna().sum()}/{len(blue)} non-null")

    # ===== NEW: Indices requiring RED band =====
    if has_red:
        # NDVI - Normalized Difference Vegetation Index
        # Key indicator of vegetation density → nutrient runoff
        out["NDVI"] = (nir - red) / (nir + red + eps)

        # NDBI - Normalized Difference Built-up Index
        # Urban/built-up area indicator → pollution proxy
        out["NDBI"] = (swir16 - nir) / (swir16 + nir + eps)

        # Red-related ratios
        out["nir_red_ratio"] = nir / (red + eps)
        out["red_green_ratio"] = red / (green + eps)
        out["red_swir16_ratio"] = red / (swir16 + eps)

        # Log red
        out["log_red"] = np.log1p(red.clip(lower=0))

    # ===== NEW: Indices requiring BLUE band =====
    if has_blue:
        # Blue-related ratios
        out["blue_green_ratio"] = blue / (green + eps)
        out["nir_blue_ratio"] = nir / (blue + eps)

        # Log blue
        out["log_blue"] = np.log1p(blue.clip(lower=0))

    # ===== NEW: Indices requiring BOTH red and blue =====
    if has_red and has_blue:
        # EVI - Enhanced Vegetation Index
        # Better than NDVI in high-biomass areas, corrects for atmospheric effects
        evi_denom = nir + 6.0 * red - 7.5 * blue + 1.0
        out["EVI"] = 2.5 * (nir - red) / (evi_denom + eps)

        # Turbidity proxy (Blue / Red ratio)
        # Higher values = clearer water, lower = more turbid/sediment
        out["turbidity_proxy"] = blue / (red + eps)

        # BSI - Bare Soil Index
        # Exposed soil near water bodies → erosion/sedimentation
        bsi_num = (swir16 + red) - (nir + blue)
        bsi_den = (swir16 + red) + (nir + blue)
        out["BSI"] = bsi_num / (bsi_den + eps)

        # AWEI - Automated Water Extraction Index (shadow version)
        # Water body detection that handles shadows
        out["AWEI"] = blue - 2.5 * green - 1.5 * (nir + swir16) - 0.25 * swir22

        # Red-Blue difference (sediment indicator)
        out["red_blue_diff"] = red - blue
        out["red_blue_ratio"] = red / (blue + eps)

    # ===== Existing indices (from v6) =====
    # Normalized Difference Indices
    out["NDWI"] = (green - nir) / (green + nir + eps)
    out["NDMI"] = (nir - swir16) / (nir + swir16 + eps)
    out["MNDWI"] = (green - swir16) / (green + swir16 + eps)
    out["MI2"] = (nir - swir22) / (nir + swir22 + eps)
    out["MNDWI2"] = (green - swir22) / (green + swir22 + eps)
    out["SWIR_NDI"] = (swir16 - swir22) / (swir16 + swir22 + eps)

    # Band Ratios
    out["nir_green_ratio"] = nir / (green + eps)
    out["nir_swir16_ratio"] = nir / (swir16 + eps)
    out["nir_swir22_ratio"] = nir / (swir22 + eps)
    out["green_swir16_ratio"] = green / (swir16 + eps)
    out["green_swir22_ratio"] = green / (swir22 + eps)
    out["swir16_swir22_ratio"] = swir16 / (swir22 + eps)

    # Band Statistics (include red/blue if available)
    bands = [nir, green, swir16, swir22]
    if has_red:
        bands.append(red)
    if has_blue:
        bands.append(blue)
    band_stack = np.column_stack(bands)
    out["band_mean"] = np.nanmean(band_stack, axis=1)
    out["band_std"] = np.nanstd(band_stack, axis=1)
    out["band_range"] = np.nanmax(band_stack, axis=1) - np.nanmin(band_stack, axis=1)
    out["band_max"] = np.nanmax(band_stack, axis=1)
    out["band_min"] = np.nanmin(band_stack, axis=1)
    out["band_cv"] = out["band_std"] / (out["band_mean"] + eps)

    # Log-transformed bands
    out["log_nir"] = np.log1p(nir.clip(lower=0))
    out["log_green"] = np.log1p(green.clip(lower=0))
    out["log_swir16"] = np.log1p(swir16.clip(lower=0))
    out["log_swir22"] = np.log1p(swir22.clip(lower=0))

    # Interaction terms
    out["nir_x_green"] = nir * green
    out["swir16_x_swir22"] = swir16 * swir22
    out["nir_x_swir16"] = nir * swir16
    out["green_x_swir22"] = green * swir22

    return out


def engineer_terraclimate_features(df):
    """Engineer interaction features from TerraClimate variables."""
    out = df.copy()
    eps = 1e-10

    def get_col(name):
        if name in out.columns: return out[name].astype(float)
        if name + "_terra" in out.columns: return out[name + "_terra"].astype(float)
        return None

    aet = get_col("aet")
    pet = get_col("pet")
    ppt = get_col("ppt")
    defic = get_col("def")
    soil = get_col("soil")
    tmax = get_col("tmax")
    tmin = get_col("tmin")
    srad = get_col("srad")
    vpd = get_col("vpd")
    vap = get_col("vap")
    q = get_col("q")
    ws = get_col("ws")
    pdsi = get_col("PDSI")

    available = sum(1 for x in [aet,pet,ppt,defic,soil,tmax,tmin,srad,vpd,vap,q,ws,pdsi] if x is not None)
    print(f"  TerraClimate variables for engineering: {available}")

    if available < 2:
        print("  Skipping TerraClimate feature engineering (too few variables)")
        return out

    # Existing v6 features
    if pet is not None and ppt is not None:
        out["aridity_index"] = pet / (ppt + eps)
    if aet is not None and pet is not None:
        out["aet_pet_ratio"] = aet / (pet + eps)
    if ppt is not None and pet is not None:
        out["water_surplus"] = ppt - pet
    if tmax is not None and tmin is not None:
        out["temp_range"] = tmax - tmin
        out["temp_mean"] = (tmax + tmin) / 2.0
    if q is not None and ppt is not None:
        out["runoff_ratio"] = q / (ppt + eps)
    if soil is not None and defic is not None:
        out["soil_x_deficit"] = soil * defic
    if srad is not None and vpd is not None:
        out["srad_x_vpd"] = srad * vpd

    # NEW v7: Additional TerraClimate interactions
    if vap is not None and vpd is not None:
        out["vap_total"] = vap + vpd
        out["relative_humidity_proxy"] = vap / (vap + vpd + eps)
    if ws is not None and pet is not None:
        out["wind_evap_interaction"] = ws * pet
    if srad is not None and tmax is not None:
        out["radiation_temp_interaction"] = srad * tmax
    if soil is not None and ppt is not None:
        out["soil_ppt_ratio"] = soil / (ppt + eps)
    if aet is not None and ppt is not None:
        out["aet_ppt_ratio"] = aet / (ppt + eps)
    if defic is not None and ppt is not None:
        out["deficit_ppt_ratio"] = defic / (ppt + eps)
    if pdsi is not None:
        # PDSI categories: drought severity
        out["pdsi_abs"] = pdsi.abs()
        out["is_drought"] = (pdsi < -2).astype(float)
        out["is_wet"] = (pdsi > 2).astype(float)

    return out


def add_temporal_features(df, date_col="Sample Date"):
    out = df.copy()
    dt = out[date_col]
    if not pd.api.types.is_datetime64_any_dtype(dt):
        dt = parse_date(dt)
    month = dt.dt.month
    doy = dt.dt.dayofyear
    out["year"] = dt.dt.year
    out["month"] = month
    out["month_sin"] = np.sin(2 * np.pi * month / 12)
    out["month_cos"] = np.cos(2 * np.pi * month / 12)
    out["doy_sin"] = np.sin(2 * np.pi * doy / 365)
    out["doy_cos"] = np.cos(2 * np.pi * doy / 365)
    # Season encoding (Southern Hemisphere)
    season_map = {12: 0, 1: 0, 2: 0, 3: 1, 4: 1, 5: 1,
                  6: 2, 7: 2, 8: 2, 9: 3, 10: 3, 11: 3}
    out["season"] = month.map(season_map)
    return out


def add_spatial_features(df):
    """NEW v7: Add spatial proxy features."""
    out = df.copy()
    lat = out["Latitude"].astype(float)
    lon = out["Longitude"].astype(float)

    # Lat/Lon interactions → encode spatial gradients
    out["lat_x_lon"] = lat * lon
    out["lat_sq"] = lat ** 2
    out["lon_sq"] = lon ** 2

    # Distance-to-coast proxy
    # South Africa: coast is roughly at the edges of the country
    # Simple approximation: distance from nearest coastline point
    # Eastern coast is roughly lon ~30-32, southern coast lat ~ -34
    # Western coast roughly lon ~17-18
    # Interior is roughly lat -25 to -28, lon 25-30
    # Use min distance to approximate coast boundaries
    coast_dist_east = np.abs(lon - 32.0)  # distance from east coast
    coast_dist_west = np.abs(lon - 17.0)  # distance from west coast
    coast_dist_south = np.abs(lat - (-34.5))  # distance from south coast
    out["coast_dist_min"] = np.minimum(np.minimum(coast_dist_east, coast_dist_west), coast_dist_south)

    # Inland indicator (further from all coasts = more inland)
    out["inland_score"] = coast_dist_east + coast_dist_west + coast_dist_south

    return out


def add_geo_clusters(train_df, test_df, n_clusters=20, seed=42):
    train_df = train_df.copy()
    test_df  = test_df.copy()
    train_xy = train_df[["Latitude", "Longitude"]].to_numpy(dtype=float)
    test_xy  = test_df[["Latitude", "Longitude"]].to_numpy(dtype=float)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init="auto")
    cluster_labels = kmeans.fit_predict(train_xy)
    centers = kmeans.cluster_centers_
    d2 = ((test_xy[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    test_labels = d2.argmin(axis=1)
    train_df["geo_cluster"] = cluster_labels.astype(int)
    test_df["geo_cluster"]  = test_labels.astype(int)
    return train_df, test_df


def add_cluster_stats(train_df, test_df, stat_features):
    """NEW v7: Add cluster-level statistics as features.
    Captures the local environment around each sampling point."""
    train_out = train_df.copy()
    test_out = test_df.copy()

    # Only use features that exist and have non-null values
    valid_features = [f for f in stat_features if f in train_out.columns
                      and train_out[f].notna().sum() > 10]

    if not valid_features:
        print("  No valid features for cluster stats")
        return train_out, test_out

    print(f"  Computing cluster stats for: {valid_features}")

    for feat in valid_features:
        # Compute stats from training data
        cluster_mean = train_out.groupby("geo_cluster")[feat].transform("mean")
        cluster_std = train_out.groupby("geo_cluster")[feat].transform("std")

        train_out[f"{feat}_cluster_mean"] = cluster_mean
        train_out[f"{feat}_cluster_std"] = cluster_std
        # Deviation from cluster mean (how unusual is this point?)
        train_out[f"{feat}_cluster_dev"] = train_out[feat] - cluster_mean

        # For test: map cluster means/stds from training
        cluster_stats = train_out.groupby("geo_cluster")[feat].agg(["mean", "std"]).reset_index()
        cluster_stats.columns = ["geo_cluster", f"{feat}_cluster_mean", f"{feat}_cluster_std"]
        test_out = test_out.merge(cluster_stats, on="geo_cluster", how="left")
        if feat in test_out.columns:
            test_out[f"{feat}_cluster_dev"] = test_out[feat] - test_out[f"{feat}_cluster_mean"]
        else:
            test_out[f"{feat}_cluster_dev"] = np.nan

    return train_out, test_out


# ====================== LOAD DATA ======================
train_labels = pd.read_csv(TRAIN_LABELS_CSV)
landsat_train = pd.read_csv(LANDSAT_TRAIN_CSV)
landsat_val = pd.read_csv(LANDSAT_VAL_CSV)
terraclimate_train = pd.read_csv(TERRACLIMATE_TRAIN_CSV)
terraclimate_val = pd.read_csv(TERRACLIMATE_VAL_CSV)
submission_template = pd.read_csv(SUBMISSION_TEMPLATE_CSV)

print("Loaded shapes:")
print("train_labels:", train_labels.shape)
print("landsat_train:", landsat_train.shape, "columns:", list(landsat_train.columns))
print("landsat_val:", landsat_val.shape, "columns:", list(landsat_val.columns))
print("terraclimate_train:", terraclimate_train.shape)
print("terraclimate_val:", terraclimate_val.shape)

# Check for red/blue bands
has_red_train = "red" in landsat_train.columns
has_blue_train = "blue" in landsat_train.columns
has_red_val = "red" in landsat_val.columns
has_blue_val = "blue" in landsat_val.columns
print(f"\nRed band: train={has_red_train}, val={has_red_val}")
print(f"Blue band: train={has_blue_train}, val={has_blue_val}")
if has_red_train:
    print(f"  Red non-null (train): {landsat_train['red'].notna().sum()}/{len(landsat_train)}")
if has_blue_train:
    print(f"  Blue non-null (train): {landsat_train['blue'].notna().sum()}/{len(landsat_train)}")

tc_data_cols = [c for c in terraclimate_train.columns if c not in ["Latitude", "Longitude", "Sample Date"]]
print(f"\nTerraClimate data columns ({len(tc_data_cols)}): {tc_data_cols}")

# ====================== PRE-AGGREGATE MONTHLY ======================
landsat_train2 = add_month_key_and_round(landsat_train)
landsat_val2   = add_month_key_and_round(landsat_val)
terr_train2    = add_month_key_and_round(terraclimate_train)
terr_val2      = add_month_key_and_round(terraclimate_val)

landsat_train_m = landsat_train2.groupby(["lat_round","lon_round","month_ts"], as_index=False).mean(numeric_only=True)
landsat_val_m   = landsat_val2.groupby(["lat_round","lon_round","month_ts"], as_index=False).mean(numeric_only=True)
terr_train_m    = terr_train2.groupby(["lat_round","lon_round","month_ts"], as_index=False).mean(numeric_only=True)
terr_val_m      = terr_val2.groupby(["lat_round","lon_round","month_ts"], as_index=False).mean(numeric_only=True)

print("\nMonthly aggregated shapes:")
print("landsat_train_m:", landsat_train_m.shape, "terr_train_m:", terr_train_m.shape)

# ====================== PREPARE BASE TRAIN/TEST ======================
train = train_labels.copy()
train["Sample Date"] = parse_date(train["Sample Date"])
train["month_ts"] = train["Sample Date"].dt.to_period("M").dt.to_timestamp()
train["lat_round"] = train["Latitude"].round(SPATIAL_ROUND)
train["lon_round"] = train["Longitude"].round(SPATIAL_ROUND)
train["spatial_id"] = train["lat_round"].astype(str) + "_" + train["lon_round"].astype(str)

test = submission_template.copy()
test["Sample Date"] = parse_date(test["Sample Date"])
test["month_ts"] = test["Sample Date"].dt.to_period("M").dt.to_timestamp()
test["lat_round"] = test["Latitude"].round(SPATIAL_ROUND)
test["lon_round"] = test["Longitude"].round(SPATIAL_ROUND)
test["spatial_id"] = test["lat_round"].astype(str) + "_" + test["lon_round"].astype(str)

print("\nPrepared base shapes:", train.shape, test.shape)

# ====================== MERGE ======================
merge_on = ["lat_round", "lon_round", "month_ts"]

train = train.merge(landsat_train_m, how="left", on=merge_on, suffixes=("", "_landsat"))
train = train.merge(terr_train_m,    how="left", on=merge_on, suffixes=("", "_terra"))
test  = test.merge(landsat_val_m,    how="left", on=merge_on, suffixes=("", "_landsat"))
test  = test.merge(terr_val_m,       how="left", on=merge_on, suffixes=("", "_terra"))

print("\nAfter merges: train", train.shape, "test", test.shape)

# Check merge success
for name, df in [("train", train), ("test", test)]:
    landsat_nulls = df["nir"].isna().sum()
    terra_nulls = df["pet"].isna().sum() if "pet" in df.columns else -1
    print(f"  {name}: Landsat NaN={landsat_nulls}/{len(df)} ({landsat_nulls/len(df)*100:.1f}%), "
          f"TerraClimate NaN={terra_nulls}/{len(df)} ({terra_nulls/len(df)*100:.1f}%)")

# ====================== FEATURE ENGINEERING ======================
print("\n--- Landsat Feature Engineering (with RED + BLUE) ---")
train = engineer_landsat_features(train)
test = engineer_landsat_features(test)
print(f"After Landsat FE: train {train.shape}, test {test.shape}")

print("\n--- TerraClimate Feature Engineering ---")
train = engineer_terraclimate_features(train)
test = engineer_terraclimate_features(test)
print(f"After TerraClimate FE: train {train.shape}, test {test.shape}")

print("\n--- Temporal Features ---")
train = add_temporal_features(train)
test = add_temporal_features(test)
print(f"After temporal FE: train {train.shape}, test {test.shape}")

print("\n--- Spatial Features (NEW v7) ---")
train = add_spatial_features(train)
test = add_spatial_features(test)
print(f"After spatial FE: train {train.shape}, test {test.shape}")

# ====================== CLEAN UP: Replace inf with NaN ======================
print("\n--- Cleaning inf values (leaving NaN for LightGBM native handling) ---")
numeric_cols = train.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    train[col] = train[col].replace([np.inf, -np.inf], np.nan)
    test[col] = test[col].replace([np.inf, -np.inf], np.nan)

# ====================== GEO CLUSTERS ======================
train, test = add_geo_clusters(train, test, n_clusters=N_CLUSTERS, seed=SEED)

n_unique_clusters = train["geo_cluster"].nunique()
print(f"\nGeo clusters: {n_unique_clusters} (N_CLUSTERS={N_CLUSTERS})")
print(f"Cluster sizes: min={train.groupby('geo_cluster').size().min()}, "
      f"max={train.groupby('geo_cluster').size().max()}, "
      f"mean={train.groupby('geo_cluster').size().mean():.0f}")

# ====================== CLUSTER-LEVEL STATISTICS (NEW v7) ======================
print("\n--- Cluster-Level Statistics (NEW v7) ---")
# Key spectral indices to compute cluster stats for
cluster_stat_features = ["NDVI", "NDMI", "MNDWI", "NDWI", "EVI", "BSI", "NDBI",
                          "nir", "green", "swir16", "pet", "ppt", "tmax", "PDSI"]
train, test = add_cluster_stats(train, test, cluster_stat_features)
print(f"After cluster stats: train {train.shape}, test {test.shape}")

# ====================== BUILD FEATURE LIST ======================
exclude = ["Latitude", "Longitude", "Sample Date", "month_ts",
           "lat_round", "lon_round", "spatial_id", "geo_cluster",
           "Latitude_landsat", "Longitude_landsat",
           "Latitude_terra", "Longitude_terra"] + TARGETS

base_features = [c for c in train.columns
                 if c not in exclude
                 and c not in DROP_FEATURES
                 and not c.endswith("_terra")  # drop duplicate suffixed columns from merge
                 and not c.endswith("_landsat")
                 and pd.api.types.is_numeric_dtype(train[c])]

features = list(dict.fromkeys(base_features))
features = [f for f in features if f not in DROP_FEATURES]

print(f"\nTotal features: {len(features)}")
print(f"\nFeature categories:")
# Categorize features for display
landsat_raw = [f for f in features if f in ["nir","green","swir16","swir22","red","blue"]]
landsat_idx = [f for f in features if f in ["NDVI","EVI","NDBI","BSI","AWEI","turbidity_proxy",
                                            "NDWI","NDMI","MNDWI","MI2","MNDWI2","SWIR_NDI"]]
spatial_feats = [f for f in features if f in ["lat_x_lon","lat_sq","lon_sq","coast_dist_min","inland_score"]
                 or "cluster_" in f]
temporal_feats = [f for f in features if f in ["year","month","month_sin","month_cos","doy_sin","doy_cos","season"]]
print(f"  Landsat raw bands: {len(landsat_raw)} {landsat_raw}")
print(f"  Spectral indices: {len(landsat_idx)} {landsat_idx}")
print(f"  Spatial features: {len(spatial_feats)}")
print(f"  Temporal features: {len(temporal_feats)}")
print(f"  Other (ratios, stats, terra, interactions): {len(features) - len(landsat_raw) - len(landsat_idx) - len(spatial_feats) - len(temporal_feats)}")

# Diagnostics
nan_counts_train = train[features].isna().sum()
nan_features = nan_counts_train[nan_counts_train > 0]
if len(nan_features) > 0:
    print(f"\nFeatures with NaN (LightGBM handles these natively):")
    for f, cnt in list(nan_features.items())[:15]:  # show top 15
        print(f"  {f}: {cnt}/{len(train)} ({cnt/len(train)*100:.1f}%)")
    if len(nan_features) > 15:
        print(f"  ... and {len(nan_features) - 15} more")
else:
    print("\nNo NaN in features (all merges succeeded)")

# ====================== TRAINING ======================
LGB_PARAMS = dict(
    n_estimators=1500,     # Slightly more trees (was 1200) — early stopping prevents overfitting
    learning_rate=0.03,    # Lower LR (was 0.05) — more gradual learning with more features
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.7,  # Slightly lower (was 0.8) — more randomness with more features
    min_child_samples=15,  # Slightly more regularization (was 10)
    reg_alpha=0.1,         # L1 regularization (NEW)
    reg_lambda=1.0,        # L2 regularization (NEW)
    n_jobs=-1,
)

print(f"\nLGB_PARAMS: {LGB_PARAMS}")

oof = {t: np.zeros(len(train)) for t in TARGETS}
models = {t: [] for t in TARGETS}
cv_scores = {}


def train_target_multiseed(target_name, feature_cols, use_log_transform=False):
    """Train with optional log1p transform for skewed targets (DRP)."""
    y_raw = train[target_name].values.copy()

    if use_log_transform:
        y = np.log1p(np.clip(y_raw, 0, None))  # log1p transform
        print(f"  Using log1p transform for {target_name}")
        print(f"  Original range: [{y_raw.min():.2f}, {y_raw.max():.2f}]")
        print(f"  Log range: [{y.min():.2f}, {y.max():.2f}]")
    else:
        y = y_raw.copy()

    X = train[feature_cols]
    groups = train["geo_cluster"].values

    oof_preds_accum = np.zeros(len(train))
    oof_counts = np.zeros(len(train))
    all_seed_models = []
    all_fold_r2 = []

    for seed_idx, seed in enumerate(SEEDS):
        print(f"\n  --- Seed {seed} ({seed_idx+1}/{N_SEEDS}) ---")
        gkf = GroupKFold(n_splits=N_SPLITS)
        seed_models = []

        for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups)):
            Xtr, Xva = X.iloc[tr_idx], X.iloc[va_idx]
            ytr, yva = y[tr_idx], y[va_idx]

            model = LGBMRegressor(**LGB_PARAMS, random_state=seed + fold)
            model.fit(
                Xtr, ytr,
                eval_set=[(Xva, yva)],
                eval_metric="rmse",
                callbacks=[
                    lgb.early_stopping(stopping_rounds=100),
                    lgb.log_evaluation(200),
                ],
            )

            preds_va = model.predict(Xva)

            if use_log_transform:
                # Evaluate on original scale
                preds_va_orig = np.expm1(preds_va)
                preds_va_orig = np.clip(preds_va_orig, 0, None)
                r2 = r2_score(y_raw[va_idx], preds_va_orig)
                oof_preds_accum[va_idx] += preds_va  # accumulate in log space
            else:
                preds_va = np.clip(preds_va, 0, None)
                r2 = r2_score(y_raw[va_idx], preds_va)
                oof_preds_accum[va_idx] += preds_va

            oof_counts[va_idx] += 1
            all_fold_r2.append(r2)
            seed_models.append(model)
            print(f"    Fold {fold} R2: {r2:.4f} (best_iter={model.best_iteration_})")

            del Xtr, Xva, ytr, yva
            gc.collect()

        all_seed_models.append((seed_models, use_log_transform))

    oof_preds = oof_preds_accum / oof_counts

    if use_log_transform:
        oof_preds_orig = np.clip(np.expm1(oof_preds), 0, None)
        oof_r2 = r2_score(y_raw, oof_preds_orig)
    else:
        oof_preds_orig = np.clip(oof_preds, 0, None)
        oof_r2 = r2_score(y_raw, oof_preds_orig)

    cv_mean = np.mean(all_fold_r2)
    cv_std = np.std(all_fold_r2)

    print(f"\n  >> {target_name}: Pooled OOF R2 = {oof_r2:.4f}")
    print(f"  >> {target_name}: Mean fold R2 = {cv_mean:.4f} +/- {cv_std:.4f}")
    print(f"  >> {target_name}: {N_SEEDS} seeds x {N_SPLITS} folds = {len(all_fold_r2)} models")

    return oof_preds_orig, all_seed_models, (cv_mean, cv_std, oof_r2)


def predict_test_multiseed(seed_models_list, feature_cols, test_df):
    preds = np.zeros(len(test_df))
    n_models = 0
    for seed_models, use_log in seed_models_list:
        for model in seed_models:
            raw_preds = model.predict(test_df[feature_cols])
            if use_log:
                raw_preds = np.expm1(raw_preds)
            preds += raw_preds
            n_models += 1
    preds /= n_models
    preds = np.clip(preds, 0, None)
    print(f"  Test: averaged {n_models} models, range [{preds.min():.2f}, {preds.max():.2f}]")
    return preds


# ====================== FEATURE IMPORTANCE HELPER ======================
def get_top_features(seed_models_list, feature_cols, top_n=None, min_importance=0):
    """Get top features by aggregated importance across all models."""
    importance = np.zeros(len(feature_cols))
    n_models = 0
    for seed_models, _ in seed_models_list:
        for model in seed_models:
            importance += model.feature_importances_
            n_models += 1
    importance /= n_models

    feat_imp = pd.DataFrame({"feature": feature_cols, "importance": importance})
    feat_imp = feat_imp.sort_values("importance", ascending=False)

    if min_importance > 0:
        feat_imp = feat_imp[feat_imp["importance"] >= min_importance]
    if top_n:
        feat_imp = feat_imp.head(top_n)

    return feat_imp


# ====================== STAGE 1: Electrical Conductance ======================
target_ec = "Electrical Conductance"
print(f"\n{'='*60}")
print(f"=== STAGE 1: {target_ec} ===")
print(f"{'='*60}")

oof_ec, ec_seed_models, ec_cv = train_target_multiseed(target_ec, features)
oof[target_ec] = oof_ec
models[target_ec] = ec_seed_models
cv_scores[target_ec] = ec_cv

# Feature importance for EC
ec_imp = get_top_features(ec_seed_models, features, top_n=20)
print(f"\n  Top 20 features for {target_ec}:")
for _, row in ec_imp.iterrows():
    print(f"    {row['feature']:35s} {row['importance']:.1f}")

test_ec_preds = predict_test_multiseed(ec_seed_models, features, test)
test[target_ec] = test_ec_preds
train["oof_ec"] = oof_ec
test["oof_ec"] = test_ec_preds

# ====================== STAGE 2: Total Alkalinity (uses oof_ec) ======================
target_alk = "Total Alkalinity"
print(f"\n{'='*60}")
print(f"=== STAGE 2: {target_alk} (uses oof_ec) ===")
print(f"{'='*60}")

features_alk = features.copy()
if "oof_ec" not in features_alk:
    features_alk.append("oof_ec")

oof_alk, alk_seed_models, alk_cv = train_target_multiseed(target_alk, features_alk)
oof[target_alk] = oof_alk
models[target_alk] = alk_seed_models
cv_scores[target_alk] = alk_cv

# Feature importance for Alkalinity
alk_imp = get_top_features(alk_seed_models, features_alk, top_n=20)
print(f"\n  Top 20 features for {target_alk}:")
for _, row in alk_imp.iterrows():
    print(f"    {row['feature']:35s} {row['importance']:.1f}")

test_alk_preds = predict_test_multiseed(alk_seed_models, features_alk, test)
test[target_alk] = test_alk_preds
train["oof_alk"] = oof_alk
test["oof_alk"] = test_alk_preds

# ====================== STAGE 3: DRP with LOG TRANSFORM (uses oof_ec + oof_alk) ======================
target_drp = "Dissolved Reactive Phosphorus"
print(f"\n{'='*60}")
print(f"=== STAGE 3: {target_drp} (uses oof_ec + oof_alk, LOG TRANSFORM) ===")
print(f"{'='*60}")

# Show DRP distribution to justify log transform
drp_vals = train_labels[target_drp].dropna()
print(f"\n  DRP distribution (original scale):")
print(f"    min={drp_vals.min():.1f}, median={drp_vals.median():.1f}, "
      f"mean={drp_vals.mean():.1f}, max={drp_vals.max():.1f}")
print(f"    skewness={drp_vals.skew():.2f}, kurtosis={drp_vals.kurtosis():.2f}")
print(f"    % below median: {(drp_vals < drp_vals.median()).mean()*100:.1f}%")

features_drp = features.copy()
for col in ["oof_ec", "oof_alk"]:
    if col not in features_drp:
        features_drp.append(col)

oof_drp, drp_seed_models, drp_cv = train_target_multiseed(
    target_drp, features_drp, use_log_transform=True
)
oof[target_drp] = oof_drp
models[target_drp] = drp_seed_models
cv_scores[target_drp] = drp_cv

# Feature importance for DRP
drp_imp = get_top_features(drp_seed_models, features_drp, top_n=20)
print(f"\n  Top 20 features for {target_drp}:")
for _, row in drp_imp.iterrows():
    print(f"    {row['feature']:35s} {row['importance']:.1f}")

test_drp_preds = predict_test_multiseed(drp_seed_models, features_drp, test)
test[target_drp] = test_drp_preds

# ====================== OOF R2 SUMMARY ======================
print(f"\n{'='*60}")
print("OOF R2 SUMMARY (pooled, original scale):")
print(f"{'='*60}")
oof_r2s = {}
for t in TARGETS:
    r2 = r2_score(train_labels[t].values, oof[t])
    oof_r2s[t] = r2
    print(f"  {t}: {r2:.4f}")

mean_oof_r2 = np.mean(list(oof_r2s.values()))
print(f"\n  MEAN OOF R2 = {mean_oof_r2:.4f}")

# ====================== BUILD SUBMISSION ======================
submission_df = submission_template.copy()
for t in TARGETS:
    submission_df[t] = test[t].values
    submission_df[t] = submission_df[t].clip(lower=0)

submission_df.to_csv(OUT_LOCAL, index=False)
try:
    submission_df.to_csv(OUT_TMP, index=False)
except Exception:
    pass

print(f"\nSaved submission to: {OUT_LOCAL}")
print(f"Also saved to: {OUT_TMP}")

print("\nSubmission preview:")
print(submission_df.describe())

# ====================== SNOWFLAKE UPLOAD ======================
try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
    sql_cmd = "PUT file://" + OUT_TMP + " snow://workspace/USER$.PUBLIC.DEFAULT$/versions/live/ AUTO_COMPRESS=FALSE OVERWRITE=TRUE"
    session.sql(sql_cmd).collect()
    print("\nUploaded submission.csv to Snowflake workspace (live).")
except Exception as e:
    print(f"\nSnowflake PUT skipped: {e}")

# ====================== FINAL REPORT ======================
print(f"\n{'='*60}")
print("FINAL CV REPORT")
print(f"{'='*60}")
print(f"N_CLUSTERS={N_CLUSTERS}, N_SPLITS={N_SPLITS}, N_SEEDS={N_SEEDS}")
print(f"Total models per target: {N_SEEDS * N_SPLITS}")
print(f"LightGBM: lr={LGB_PARAMS['learning_rate']}, max_depth={LGB_PARAMS['max_depth']}, "
      f"n_est={LGB_PARAMS['n_estimators']}, min_child={LGB_PARAMS['min_child_samples']}")
print(f"Total features (base): {len(features)}")
print()
for t in TARGETS:
    m, s, oof_r2 = cv_scores[t]
    print(f"  {t}:")
    print(f"    Fold R2 = {m:.4f} +/- {s:.4f}")
    print(f"    OOF R2  = {oof_r2:.4f}")

print(f"\n  MEAN OOF R2 = {mean_oof_r2:.4f}")
print(f"\n  v7 changes vs v6:")
print(f"    + Red + Blue Landsat bands (NDVI, EVI, turbidity, BSI, AWEI, NDBI)")
print(f"    + Spatial proxy features (lat/lon interactions, coast distance)")
print(f"    + Cluster-level statistics (environmental context)")
print(f"    + DRP log1p transform (handles skewed distribution)")
print(f"    + Additional TerraClimate interactions (humidity, wind, PDSI categories)")
print(f"    + Lower learning rate (0.03), L1/L2 regularization")
print(f"    + More temporal features (year, month, season)")
print(f"\n  Baselines: v3=0.278, v5=0.1114, v6=0.19 (eval)")